# Day 4 v2 — Model 02: DNN + HashingVec 5000

**Architecture:** HashingVectorizer (binary, 5000 features) → PriceDNN (8 ResidualBlocks, hidden=4096)

**Target:** MAE < 85k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

**Key difference vs Model 01:** HashingVec 5000 is small enough to `.toarray()` per batch without memory pressure. Mirrors English `deep_neural_network.py` config.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch
from sklearn.feature_extraction.text import HashingVectorizer

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.deep_neural_network_sparse import SparseDNNRunner

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

HashingVectorizer với `binary=True` — encode 0/1 thay vì đếm frequency.
Không dùng `stop_words` vì tiếng Việt không có danh sách stop words chuẩn.

In [ ]:
vectorizer = HashingVectorizer(
    n_features=5_000,
    binary=True,
    # Không dùng stop_words — tiếng Việt
)

runner = SparseDNNRunner(train, val)
runner.setup(vectorizer, batch_size=256, num_blocks=8, hidden_size=4096)

## 3. Train

Max 10 epochs, early stopping patience=3.

In [ ]:
history = runner.train(epochs=10, patience=3)

## 4. Training History

In [ ]:
plot_training_history(history, title="DNN + HashingVec 5000")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/dnn_hashvec.pth")
print("Saved weights/dnn_hashvec.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/dnn_hashvec_val.json", "w") as f:
    json.dump(val_preds, f)

print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/dnn_hashvec_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

## 6. Evaluate on 200 Test Samples

In [ ]:
def dnn_hashvec_pricer(item):
    return runner.inference(item)

results = evaluate(dnn_hashvec_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R²: {results['r2']:.1f}%")

## 7. Sanity Check — Load Roundtrip

In [ ]:
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

runner.load("weights/dnn_hashvec.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")